# zReg — CPD Registration Tutorial

This notebook demonstrates Coherent Point Drift (CPD) registration using zReg. It covers
both **rigid** and **non-rigid** CPD: a synthetic ground-truth transform is applied to a
point cloud so the recovered registration can be compared against the known truth. The
notebook requires a `.tracklets` file — set `DATA_PATH` below to your local copy.

In [ ]:
DATA_PATH = "# Replace with the path to your .tracklets file"

import zreg
import copy
import open3d as o3d
import numpy as np
import torch

## Visualisation helpers

Two helper functions are defined here. `draw_registration_result` renders a source and
target point cloud side-by-side in gold and blue respectively, making it easy to assess
alignment quality. `draw_single_pc` renders a single point cloud in either source
(gold) or target (blue) colouring — useful for inspecting individual frames or
transformed results in isolation.

In [ ]:
def draw_registration_result(source, target):
    # deep copies used to avoid overwriting
    if isinstance(source, zreg.dataset.zRegPointCloud):
        # convert to open3d point cloud for visualization
        spc = zreg.dataset.zreg_to_open3d(copy.deepcopy(source))
    else:
        spc = copy.deepcopy(source)
    if isinstance(target, zreg.dataset.zRegPointCloud):
        tpc = zreg.dataset.zreg_to_open3d(copy.deepcopy(target))
    else:
        tpc = copy.deepcopy(target)
    spc.paint_uniform_color([1, 0.706, 0])
    tpc.paint_uniform_color([0, 0.651, 0.929])
    o3d.visualization.draw_plotly([spc.to_legacy(), tpc.to_legacy()])

def draw_single_pc(pc, coloring_type='source'):
    if coloring_type == 'source':
        color = [1, 0.706, 0]
    elif coloring_type == 'target':
        color = [0, 0.651, 0.929]
    else:
        raise ValueError("coloring_type must be 'source' or 'target'")
    # deep copies used to avoid overwriting
    if isinstance(pc, zreg.dataset.zRegPointCloud):
        # convert to open3d point cloud for visualization
        spc = zreg.dataset.zreg_to_open3d(copy.deepcopy(pc))
    else:
        spc = copy.deepcopy(pc)
    spc.paint_uniform_color([1, 0.706, 0])
    o3d.visualization.draw_plotly([spc.to_legacy()])

## Load trajectory data

`load_data_from_tracklets` reads a `.tracklets` file and returns a list of per-frame
point cloud objects. We load the full trajectory here; subsequent cells work with
specific frames selected by index.

In [ ]:
pcs, _alltracklets = zreg.dataset.load_data_from_tracklets(
    filepath=DATA_PATH,
    device="cpu",
)

## Rigid CPD registration

To test rigid CPD, we apply a known synthetic rotation (30 degrees around the Z axis,
expressed as a 4x4 homogeneous matrix) to a copy of `pcs[0]`. After downsampling both
source and target to the same number of points, the CPD algorithm should recover a
rotation and translation close to the applied transform. The initial misalignment is
shown by rendering source and target before registration.

In [ ]:
true_transform = np.asarray(
    [[0.866, -0.5, 0.0, 0.0], [0.5, 0.866, 0.0, 0.0], [0.0, 0.0, 1.0, 0.0], [0.0, 0.0, 0.0, 1.0]]
)

source_idx = 0
target_idx = 0

# copy the point clouds so they dont need to re-loaded
source = copy.deepcopy(pcs[source_idx])
target = copy.deepcopy(pcs[target_idx])

# downsample whichever is larger, if the same, they are returned as is
target, source = zreg.downsampling.farthest_point_down_sample(target, source)

# apply the transform to the source
source = zreg.transforms.transform_points_homogeneous(source, true_transform, return_o3d=False)
pcsource = zreg.dataset.torch_to_open3d(source)
pctarget = zreg.dataset.torch_to_open3d(target)
draw_registration_result(pcsource, pctarget)

## Run rigid CPD

`registration_cpd` runs the CPD algorithm with the specified transform type. With
`tf_type_name="rigid"` it estimates a rotation matrix and translation vector that
aligns `source` to `target`. The result object exposes the recovered transformation
via `result.transformation.rot` and `result.transformation.t`.

In [ ]:
# Do a rigid registration. If another type of registration is desired, then call it
result = zreg.cpd.registration_cpd(
    source=copy.deepcopy(source),
    target=copy.deepcopy(target),
    tf_type_name="rigid",
    use_color=False,
)
# print(result)
print(f"Rotation:\n{result.transformation.rot}\nTranslation:\n{result.transformation.t}")

## Visualise rigid CPD result

We apply the recovered rigid transformation to fresh copies of source and target,
then render them together. Good alignment (source and target overlapping) confirms
that CPD successfully recovered the synthetic rotation.

In [ ]:
# Plot results
source2 = copy.deepcopy(pcs[source_idx])
target2 = copy.deepcopy(pcs[target_idx])

source2["pos"] = result.transformation.transform(source2["pos"])

pc1new = zreg.dataset.torch_to_open3d(source2)
pc0new = zreg.dataset.torch_to_open3d(target2)
draw_registration_result(pc1new, pc0new)

## Non-rigid CPD registration

`NonRigidTransformation` creates a smooth, spatially-varying deformation field controlled
by a Gaussian kernel with bandwidth `beta`. Here we construct a non-rigid deformation
from frame `pcs[0]` using uniform weights, apply it to a downsampled copy, and render
the result to visualise the deformation before running non-rigid CPD.

In [ ]:
# example non-rigid transformation + example of how the other transformations work
x = copy.deepcopy(pcs[0])
y = copy.deepcopy(pcs[0])
x, y = zreg.downsampling.farthest_point_down_sample(x, y)

NonRigidTrans = zreg.transforms.NonRigidTransformation(
    w=torch.ones_like(pcs[0]["pos"]), points=pcs[0]["pos"], beta=0.005
)

y["pos"] = NonRigidTrans.transform(y["pos"])

pc1new = zreg.dataset.torch_to_open3d(x)
pc0new = zreg.dataset.torch_to_open3d(y)

draw_registration_result(pc1new, pc0new)

## Run non-rigid CPD

Non-rigid CPD (`tf_type_name="nonrigid_constrained"`) estimates a spatially-varying
deformation field rather than a single global matrix. The `beta` parameter controls
the smoothness of the deformation and `lmd` is a regularisation weight. After
convergence, `result.transformation.g` contains the Gram matrix of the kernel.

In [ ]:
# example non-rigid registration
result = zreg.cpd.registration_cpd(
    source=copy.deepcopy(x),
    target=copy.deepcopy(y),
    tf_type_name="nonrigid_constrained",
    use_color=False,
    maxiter=100,
    beta=0.01,
    lmd=0.1,
)
print(result)

## Visualise non-rigid CPD result

We apply the recovered non-rigid transformation to fresh copies of `x` and `y` and
render the aligned result. Note: the original call passes a third positional argument
(`None`) to `draw_registration_result`, which only accepts two parameters — the extra
argument is silently ignored by Python and has no effect on the output.

In [ ]:
print(result.transformation.g)

x1 = copy.deepcopy(x)
y1 = copy.deepcopy(y)

x1["pos"] = result.transformation.transform(x1["pos"])

pc1new = zreg.dataset.torch_to_open3d(x1)
pc0new = zreg.dataset.torch_to_open3d(y1)
# Note: draw_registration_result only accepts 2 positional args; None is ignored
draw_registration_result(pc1new, pc0new)